In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
from scipy.signal import correlate

merged = pd.read_csv("../data/processed/all_cities_combined.csv")
merged["week"] = pd.to_datetime(merged["week"])

CITIES = ["bangkok", "krakow", "melbourne", "lahore"]
ALL_TERMS = ["air_quality", "air_purifier", "N95_mask", "hospital",
             "breathing_difficulty", "cough_medicine", "eye_drops", "chest_pain"]
TERM_CATEGORY = {
    "air_quality":          "awareness",
    "air_purifier":         "mitigation",
    "N95_mask":             "mitigation",
    "hospital":             "health-seeking",
    "breathing_difficulty": "health-seeking",
    "cough_medicine":       "health-seeking",
    "eye_drops":            "health-seeking",
    "chest_pain":           "health-seeking"
}
MAX_LAG = 8

def cross_corr(df, term, max_lag=MAX_LAG):
    pm25_norm = (df["pm25_mean"] - df["pm25_mean"].mean()) / df["pm25_mean"].std()
    srch_norm = (df[term] - df[term].mean()) / df[term].std()
    cc = correlate(srch_norm, pm25_norm, mode="full")
    lags = np.arange(-(len(pm25_norm)-1), len(pm25_norm))
    mask = (lags >= -max_lag) & (lags <= max_lag)
    return lags[mask], cc[mask] / len(pm25_norm)

print("Loaded ✓")
print(f"Shape: {merged.shape}")
print(merged.groupby("city").size())

In [ ]:
N_SPIKES = 3
WINDOW = 4
MERGE_WEEKS = 3

spike_rows = []
fig, axes = plt.subplots(len(CITIES), N_SPIKES, figsize=(16, 3.6 * len(CITIES)))

for row, city in enumerate(CITIES):
    df = merged[merged["city"] == city].sort_values("week").reset_index(drop=True)

    spike_weeks = []
    for week in df.sort_values("pm25_mean", ascending=False)["week"]:
        if all(abs(week - chosen) > pd.Timedelta(weeks=MERGE_WEEKS) for chosen in spike_weeks):
            spike_weeks.append(week)
        if len(spike_weeks) == N_SPIKES:
            break

    top = df[df["week"].isin(spike_weeks)].sort_values("pm25_mean", ascending=False)
    spike_rows.append(top[["city", "week", "pm25_mean"] + ALL_TERMS])

    for col in range(N_SPIKES):
        ax = axes[row][col]
        if col >= len(top):
            ax.axis("off")
            continue
        spike_week = top["week"].iloc[col]
        window = df[(df["week"] >= spike_week - pd.Timedelta(weeks=WINDOW)) &
                    (df["week"] <= spike_week + pd.Timedelta(weeks=WINDOW))]

        ax2 = ax.twinx()
        ax.plot(window["week"], window["pm25_mean"],
                color="#d62728", linewidth=1.8, marker="o", markersize=4,
                label="PM2.5 (µg/m³)")
        ax2.plot(window["week"], window["air_quality"],
                 color="#1f77b4", linewidth=1.8, linestyle="--", marker="s", markersize=4,
                 label="Search: 'air quality'")
        ax.axvline(spike_week, color="black", linestyle=":", linewidth=1, alpha=0.6)

        ax.set_title(f"{city.capitalize()} — {spike_week.date()} "
                     f"({top['pm25_mean'].iloc[col]:.1f} µg/m³)",
                     fontsize=10, fontweight="bold")
        ax.set_ylabel("PM2.5 (µg/m³)", color="#d62728")
        ax2.set_ylabel("Search Index (0–100)", color="#1f77b4")
        ax.tick_params(axis="y", labelcolor="#d62728")
        ax2.tick_params(axis="y", labelcolor="#1f77b4")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.grid(alpha=0.3)

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=7)

spike_table = pd.concat(spike_rows, ignore_index=True)
spike_table["pm25_mean"] = spike_table["pm25_mean"].round(1)

print("=== TOP 3 PM2.5 SPIKE EVENTS PER CITY ===\n")
print(spike_table.to_string(index=False))

plt.suptitle(f"PM2.5 Spike Events: ±{WINDOW} Week Window",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/figures/spike_events.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/spike_events.png")

In [ ]:
CATEGORY_COLORS = {
    "awareness":      "#1f77b4",
    "mitigation":     "#ff7f0e",
    "health-seeking": "#2ca02c"
}

rows = []
for term in ALL_TERMS:
    peak_lags, weights = [], []
    for city in CITIES:
        df = merged[merged["city"] == city].dropna().sort_values("week")
        if len(df) <= MAX_LAG or df[term].std() == 0:
            continue
        lags, cc = cross_corr(df, term)
        peak_lags.append(lags[np.argmax(cc)])
        weights.append(abs(cc.max()))
    if len(peak_lags) == 0 or sum(weights) == 0:
        continue
    rows.append({
        "Term": term,
        "Category": TERM_CATEGORY[term],
        "Weighted Peak Lag (weeks)": round(float(np.average(peak_lags, weights=weights)), 2),
        "Mean |peak r|": round(float(np.mean(weights)), 3),
        "Cities": len(peak_lags)
    })

cascade = pd.DataFrame(rows).sort_values("Weighted Peak Lag (weeks)", ascending=False).reset_index(drop=True)

print("=== BEHAVIOURAL CASCADE: WEIGHTED AVERAGE PEAK LAG ===\n")
print(cascade.sort_values("Weighted Peak Lag (weeks)").to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 6))

colors = [CATEGORY_COLORS[cat] for cat in cascade["Category"]]
ax.barh(cascade["Term"], cascade["Weighted Peak Lag (weeks)"], color=colors, edgecolor="white")

for lag, term in zip(cascade["Weighted Peak Lag (weeks)"], cascade["Term"]):
    ax.text(lag + (0.15 if lag >= 0 else -0.15), term, f"{lag:+.1f}",
            va="center", ha="left" if lag >= 0 else "right", fontsize=9)

ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlim(-MAX_LAG - 1, MAX_LAG + 1)
ax.set_xticks(range(-MAX_LAG, MAX_LAG + 1))
ax.set_xlabel("Weighted average peak lag (weeks)\n← search leads | pollution leads →")
ax.set_ylabel("Search term")
ax.set_title("Behavioural Cascade: Peak Lag per Term (city average, weighted by |peak r|)",
             fontsize=13, fontweight="bold")
ax.legend(handles=[Patch(color=color, label=cat) for cat, color in CATEGORY_COLORS.items()],
          title="Category", fontsize=9)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/behavioural_cascade.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/behavioural_cascade.png")

In [ ]:
r_matrix = np.full((len(CITIES), len(ALL_TERMS)), np.nan)
lag_matrix = np.full((len(CITIES), len(ALL_TERMS)), np.nan)

for i, city in enumerate(CITIES):
    df = merged[merged["city"] == city].dropna().sort_values("week")
    for j, term in enumerate(ALL_TERMS):
        if len(df) <= MAX_LAG or df[term].std() == 0:
            continue
        r, p = stats.pearsonr(df["pm25_mean"], df[term])
        lags, cc = cross_corr(df, term)
        r_matrix[i, j] = r
        lag_matrix[i, j] = lags[np.argmax(cc)]

city_labels = [city.capitalize() for city in CITIES]

print("=== PEARSON r ===\n")
print(pd.DataFrame(r_matrix, index=city_labels, columns=ALL_TERMS).round(3).to_string())
print("\n=== PEAK LAG (weeks) ===\n")
print(pd.DataFrame(lag_matrix, index=city_labels, columns=ALL_TERMS).to_string())

fig, axes = plt.subplots(1, 2, figsize=(20, 5), layout="constrained")

panels = [
    (axes[0], r_matrix, "RdBu_r", -1, 1, "Pearson r", "{:+.2f}"),
    (axes[1], lag_matrix, "viridis", -MAX_LAG, MAX_LAG, "Peak lag (weeks)", "{:+.0f}")
]

for ax, matrix, cmap, vmin, vmax, label, fmt in panels:
    ax.set_facecolor("#e6e6e6")
    im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    for i in range(len(CITIES)):
        for j in range(len(ALL_TERMS)):
            value = matrix[i, j]
            if np.isnan(value):
                ax.text(j, i, "n/a", ha="center", va="center", fontsize=10, color="grey")
                continue
            red, green, blue, alpha = im.cmap(im.norm(value))
            luminance = 0.299 * red + 0.587 * green + 0.114 * blue
            ax.text(j, i, fmt.format(value), ha="center", va="center", fontsize=10,
                    color="black" if luminance > 0.5 else "white")
    ax.set_xticks(range(len(ALL_TERMS)))
    ax.set_xticklabels(ALL_TERMS, rotation=30, ha="right")
    ax.set_yticks(range(len(CITIES)))
    ax.set_yticklabels(city_labels)
    ax.set_title(label, fontweight="bold")
    fig.colorbar(im, ax=ax, label=label)

fig.suptitle("Multi-City Summary: PM2.5 vs Search Terms (4 cities × 8 terms)",
             fontsize=13, fontweight="bold")
plt.savefig("../outputs/figures/summary_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved → outputs/figures/summary_matrix.png")